In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from matplotlib.path import Path
import matplotlib.ticker as ticker
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import torch
import geopandas as gpd

plt.rcParams['font.family'] = 'Arial'
font_size = 20
plt.rcParams['font.size'] = font_size
plt.rcParams['xtick.labelsize'] = font_size
plt.rcParams['ytick.labelsize'] = font_size
plt.rcParams['legend.fontsize'] = font_size
plt.rcParams['axes.titlesize'] = font_size
plt.rcParams['axes.labelsize'] = font_size
plt.rcParams['axes.unicode_minus'] = False

## 1. Figure 2a

In [2]:
def read_crps_data(model, n):
    if model == 'DRUM':
        filename = f"plot_data/CRPS/DRUM_top{n}.csv"
    else:
        filename = f"plot_data/CRPS/LSTM_{model}_top{n}.csv"
    df = pd.read_csv(filename)
    return df['CRPS']

percentiles = [10, 5, 2, 1, 0.5, 0.2, 0.1]
models = ['DRUM', 'p', 'd']
colors = ['#E94560', '#0066CC', '#16A085']
legends = ['DRUM', 'LSTM-p', 'LSTM-d']
positions = np.arange(len(percentiles))
width = 0.15
gap = 0.05

fig, ax = plt.subplots(figsize=(13, 7))

for i, (model, color) in enumerate(zip(models, colors)):
    data = [read_crps_data(model, n) for n in percentiles]
    bp = ax.boxplot(data, positions=positions + i*(width+gap), widths=width, 
                    patch_artist=True, showfliers=False, whis=1.5,
                    medianprops={'color': color, 'linewidth': 2},
                    boxprops={'facecolor': color, 'edgecolor': 'none', 'alpha': 0.6},
                    whiskerprops={'color': color, 'linewidth': 2, 'alpha': 0.6},
                    capprops={'visible': False})
    
ax.set_xlabel('Top percentiles of peak flow (%)')
ax.set_ylabel('CRPS (mm/day)')
ax.set_xticks(positions + width)
ax.set_xticklabels(percentiles)

ax.grid(axis='y', which="both", ls="-", alpha=0.2)
ax.legend([plt.Rectangle((0,0),1,1,fc=c, ec="none", alpha=0.6) for c in colors],
          legends, loc='upper left', frameon=False)

plt.tight_layout()

figures_dir = os.path.join("Figure_2")
os.makedirs(figures_dir, exist_ok=True) 
save_path = os.path.join(figures_dir, "Figure_2a_CRPS_comparison_boxplot.jpg")
plt.savefig(save_path, dpi=600, bbox_inches="tight")
# plt.show()
plt.close()

## 2. Figure 2b

In [3]:
us_shapefile_path = "plot_data/shp/US.shp"
basin_shapefile_path = "plot_data/shp/basin_671_1984.shp"
basin_list_path = "plot_data/basin_list.txt"

us_gdf = gpd.read_file(us_shapefile_path)
basin_gdf = gpd.read_file(basin_shapefile_path)

with open(basin_list_path, 'r') as file:
    basin_list = [int(basin_id.strip()) for basin_id in file.read().splitlines()]

filtered_basin_gdf = basin_gdf[basin_gdf['hru_id'].isin(basin_list)]
filtered_basin_gdf = gpd.GeoDataFrame(
    filtered_basin_gdf, 
    geometry=gpd.points_from_xy(filtered_basin_gdf['lon_cen'], filtered_basin_gdf['lat_cen']),
    crs='EPSG:4326')

### DRUM vs. LSTM-p

In [4]:
lstmp_df = pd.read_csv("plot_data/CRPS/LSTM_p_top0.1.csv")
drum_df = pd.read_csv("plot_data/CRPS/DRUM_top0.1.csv")

lstmp_df['Basin'] = lstmp_df['Basin'].astype(int)
drum_df['Basin'] = drum_df['Basin'].astype(int)

merged_df = pd.merge(lstmp_df, drum_df, on='Basin', suffixes=('_lstm-p', '_drum'))
merged_df['deltaCRPS'] = merged_df['CRPS_lstm-p'] - merged_df['CRPS_drum']
merged_gdf = filtered_basin_gdf.merge(merged_df, left_on='hru_id', right_on='Basin')

merged_gdf['deltaCRPS_truncated'] = merged_gdf['deltaCRPS'].clip(lower=-10, upper=10)


fig, ax = plt.subplots(figsize=(18, 16))

us_gdf.plot(ax=ax, edgecolor='black', facecolor='none', linewidth=0.5)
merged_gdf = merged_gdf.sort_values('deltaCRPS_truncated')

total_basins = len(merged_df)
positive_deltas = (merged_df['deltaCRPS'] > 0).sum()
negative_deltas = (merged_df['deltaCRPS'] < 0).sum()
zero_deltas = (merged_df['deltaCRPS'] == 0).sum()

print(f"\n分析结果:")
print(f"总流域数: {total_basins}")
print(f"正值(DRUM优于LSTM-p)比例: {positive_deltas/total_basins:.2%}")
print(f"负值(DRUM劣于LSTM-p)比例: {negative_deltas/total_basins:.2%}")
if zero_deltas > 0:
    print(f"零值比例: {zero_deltas/total_basins:.2%}")

mean_delta = merged_df['deltaCRPS'].mean()
median_delta = merged_df['deltaCRPS'].median()
print(f"\n统计量:")
print(f"deltaCRPS平均值: {mean_delta:.4f}")
print(f"deltaCRPS中位数: {median_delta:.4f}")

sc = ax.scatter(merged_gdf.geometry.x, merged_gdf.geometry.y, c=merged_gdf['deltaCRPS_truncated'], 
                s=180, vmin=-10, vmax=10, cmap='RdBu', edgecolors='gray', linewidth=0.75)

highlight_basins = [11473900, 10336660, 14182500, 14138800, 2112360, 3237500, 2064000, 14400000]
highlight_gdf = merged_gdf[merged_gdf['hru_id'].isin(highlight_basins)]

verts = [(0, -1), (0.6, 0), (0, 1), (-0.6, 0), (0, -1)]
codes = [Path.MOVETO] + [Path.LINETO]*3 + [Path.CLOSEPOLY]
flat_diamond = Path(verts, codes)

ax.scatter(highlight_gdf.geometry.x, highlight_gdf.geometry.y, 
           c=highlight_gdf['deltaCRPS_truncated'], s=400, marker=flat_diamond, 
           edgecolor='red', linewidth=2.5, zorder=5, cmap='RdBu', vmin=-10, vmax=10)

ax.set_facecolor('white')
ax.set_xlim(-126, -65)
ax.set_ylim(24, 50)

cbar = fig.colorbar(sc, ax=ax, orientation='horizontal', pad=0.12, aspect=40, shrink=0.65)
cbar.set_label(r'$\Delta$CRPS (mm/day)')
cbar.ax.tick_params(labelsize=20)
cbar.set_ticks([-10, -5, 0, 5, 10])

for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_xticks([])
ax.set_yticks([])

left, bottom, width, height = 0.21, 0.29, 0.15, 0.15
hist_ax = fig.add_axes([left, bottom, width, height])
bins = np.linspace(-10, 10, 21)
hist_ax.hist(merged_gdf['deltaCRPS_truncated'], bins=bins, color='#4682B4', edgecolor='black', alpha=0.75)
hist_ax.set_xlabel(r'$\Delta$CRPS (mm/day)')
hist_ax.set_ylabel('Number of basins')
hist_ax.set_xlim(-10, 10)
hist_ax.set_ylim(0, 90)
hist_ax.set_xticks(np.arange(-10, 11, 5))
hist_ax.set_yticks([0, 40, 80])
hist_ax.tick_params(axis='both', which='major')
hist_ax.spines['right'].set_visible(False)
hist_ax.spines['top'].set_visible(False)

hist_ax.axvline(x=0, color='red', linestyle=(0, (5, 3)), linewidth=1.5, ymin=0, ymax=1.05)

figures_dir = os.path.join("Figure_2")
os.makedirs(figures_dir, exist_ok=True) 
save_path = os.path.join(figures_dir, "Figure_2b_Spatial_delta_CRPS.jpg")
plt.savefig(save_path, dpi=600, bbox_inches="tight", pad_inches=0.05)
# plt.show()
plt.close()


分析结果:
总流域数: 531
正值(DRUM优于LSTM-p)比例: 72.32%
负值(DRUM劣于LSTM-p)比例: 27.68%

统计量:
deltaCRPS平均值: 3.8286
deltaCRPS中位数: 2.1474


## Figure 2c-j

In [5]:
def crps_from_empirical_cdf(truth: torch.Tensor, ensemble: torch.Tensor) -> torch.Tensor:
    y = truth
    n = ensemble.shape[0]
    ensemble, _ = torch.sort(ensemble, dim=0)
    ans = 0
    val = ensemble[0] - y
    ans += torch.where(val > 0, val, 0.0)
    for i in range(n - 1):
        x0 = ensemble[i]
        x1 = ensemble[i + 1]
        cdf = (i + 1) / n
        val = (x1 - x0) * (cdf - 1) ** 2
        mask = y < x0
        ans += torch.where(mask, val, 0.0)
        val = (y - x0) * cdf**2 + (x1 - y) * (cdf - 1) ** 2
        mask = (y >= x0) & (y <= x1)
        ans += torch.where(mask, val, 0.0)
        mask = y > x1
        val = (x1 - x0) * cdf**2
        ans += torch.where(mask, val, 0.0)
    val = y - ensemble[-1]
    ans += torch.where(val > 0, val, 0.0)
    return ans

In [6]:
def plot_runoff_prediction(basin_id, confidence_interval=0.95, start_date='1996-06-26', start_plot_date=None, end_plot_date=None):
    # Load data
    lstm_results = pd.read_csv("plot_data/results_nowcasting/LSTM-d/all_results.csv", dtype={'Basin': str})
    lstm_basin_data = lstm_results[lstm_results['Basin'] == basin_id]

    drum_base_dir = "plot_data/results_nowcasting/DRUM"
    drum_predict_data = np.load(os.path.join(drum_base_dir, basin_id, 'predict.npy'))
    true_data = np.load(os.path.join(drum_base_dir, basin_id, 'true.npy'))

    basin_id_padded = basin_id.zfill(8)  # Ensure 8-digit USGS identifier
    aldm_sample_path = f"plot_data/results_nowcasting/LSTM-p/{basin_id_padded}/sample.npy"
    aldm_samples = np.load(aldm_sample_path)
    
    # Process date range
    start_date = datetime.strptime(start_date, '%Y-%m-%d')
    date_range = [start_date + timedelta(days=i) for i in range(len(true_data))]
    start_index = next((i for i, date in enumerate(date_range) if date >= datetime.strptime(start_plot_date, '%Y-%m-%d')), 0) if start_plot_date else 0
    end_index = next((i for i, date in enumerate(date_range) if date > datetime.strptime(end_plot_date, '%Y-%m-%d')), len(true_data)) if end_plot_date else len(true_data)
    
    # Prepare plot data
    true_data_plot = true_data[start_index:end_index]
    drum_predict_data_plot = drum_predict_data[start_index:end_index]
    lstm_basin_data_plot = lstm_basin_data.iloc[start_index:end_index]
    date_range_plot = date_range[start_index:end_index]
    aldm_samples_plot = aldm_samples[:, start_index:end_index]

    # DRUM
    drum_predict_mean = np.mean(drum_predict_data_plot, axis=1)
    drum_lower_percentile, drum_upper_percentile = np.percentile(drum_predict_data_plot, 
                                                                 [(1 - confidence_interval) / 2 * 100, 
                                                                  (1 + confidence_interval) / 2 * 100], axis=1)
    # ALDM
    aldm_mean = np.mean(aldm_samples_plot, axis=0)
    aldm_lower_percentile, aldm_upper_percentile = np.percentile(aldm_samples_plot, 
                                                                 [(1 - confidence_interval) / 2 * 100, 
                                                                  (1 + confidence_interval) / 2 * 100], axis=0)
    # Calculate CRPS
    true_tensor = torch.tensor(true_data_plot)
    
    # DRUM CRPS
    drum_ensemble = torch.tensor(drum_predict_data_plot.T)  # Shape: (50, 16)
    drum_crps = crps_from_empirical_cdf(true_tensor, drum_ensemble)
    drum_crps_mean = torch.mean(drum_crps).item()
    
    # LSTM CRPS
    # Since we don't have ensemble for LSTM, we'll create a simple ensemble by adding small random noise
    lstm_pred = torch.tensor(lstm_basin_data_plot['y_pred'].values)
    lstm_ensemble = lstm_pred.unsqueeze(0).repeat(100, 1) + torch.randn(100, len(lstm_pred)) * 0.1 * lstm_pred
    lstm_crps = crps_from_empirical_cdf(true_tensor, lstm_ensemble)
    lstm_crps_mean = torch.mean(lstm_crps).item()
    
    # ALDM CRPS
    aldm_ensemble = torch.tensor(aldm_samples_plot)  # Shape: (50, 16)
    aldm_crps = crps_from_empirical_cdf(true_tensor, aldm_ensemble)
    aldm_crps_mean = torch.mean(aldm_crps).item()
    
    # Plot
    fig, ax = plt.subplots(figsize=(5, 4), dpi=600)
    colors = {'observation': '#000000',
        'lstm': '#16A085',  # Green
        'drum': '#E94560',  # Red
        'aldm': '#0066CC'   } # Blue  
    
    ax.fill_between(date_range_plot, aldm_lower_percentile, aldm_upper_percentile, color=colors['aldm'], alpha=0.25, edgecolor='none')
    ax.fill_between(date_range_plot, drum_lower_percentile, drum_upper_percentile, color=colors['drum'], alpha=0.45, edgecolor='none')
    
    ax.plot(date_range_plot, true_data_plot, color=colors['observation'], linewidth=2)
    ax.plot(date_range_plot, lstm_basin_data_plot['y_pred'].values, color=colors['lstm'], linewidth=2)
    ax.plot(date_range_plot, aldm_mean, color=colors['aldm'], linewidth=2)
    ax.plot(date_range_plot, drum_predict_mean, color=colors['drum'], linewidth=2)
    ax.set_ylabel('Streamflow (mm/day)')
    
    interval = max(1, (date_range_plot[-1] - date_range_plot[0]).days // 2)
    ax.xaxis.set_major_locator(mdates.DayLocator(interval=interval))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.xticks(rotation=0, fontsize=20)
    plt.yticks(fontsize=20)
    
    y_min = min(np.min(true_data_plot), np.min(drum_lower_percentile), np.min(lstm_basin_data_plot['y_pred'].values), np.min(aldm_lower_percentile))
    y_max = max(np.max(true_data_plot), np.max(drum_upper_percentile), np.max(lstm_basin_data_plot['y_pred'].values), np.max(aldm_upper_percentile))
    ax.set_ylim(y_min - 0.05 * (y_max - y_min), y_max + 0.05 * (y_max - y_min))
    
    # Add CRPS results to the plot
    crps_text = f"CRPS\nDRUM: {drum_crps_mean:.2f}\nLSTM-p: {aldm_crps_mean:.2f}\nLSTM-d: {lstm_crps_mean:.2f}"
    ax.text(0.02, 0.98, crps_text, transform=ax.transAxes, fontsize=20, verticalalignment='top')
    
    plt.tight_layout()
    
    figures_dir = os.path.join("Figure_2")
    os.makedirs(figures_dir, exist_ok=True) 
    save_path = os.path.join(figures_dir, f'Figure_2c_{basin_id}.jpg')
    plt.savefig(save_path, dpi=600, bbox_inches='tight', pad_inches=0.05)
    # plt.show()
    plt.close()

In [7]:
plot_runoff_prediction('11473900', confidence_interval=0.95, start_plot_date='1996-12-23', end_plot_date='1997-01-07')

In [8]:
plot_runoff_prediction('10336660', confidence_interval=0.95,  start_plot_date='1996-12-23',end_plot_date='1997-1-07')

In [9]:
plot_runoff_prediction('14182500', confidence_interval=0.95, start_plot_date='1999-11-19', end_plot_date='1999-11-29')

In [10]:
plot_runoff_prediction('14138800', confidence_interval=0.95, start_plot_date='1999-11-19', end_plot_date='1999-11-29')

In [11]:
plot_runoff_prediction('02112360', confidence_interval=0.95, start_plot_date='2004-09-2', end_plot_date='2004-09-12')

In [12]:
plot_runoff_prediction('03237500', confidence_interval=0.95, start_plot_date='1997-02-24', end_plot_date='1997-3-6')

In [13]:
plot_runoff_prediction('14400000', confidence_interval=0.95, start_plot_date='1996-11-13', end_plot_date='1996-11-23')

In [14]:
plot_runoff_prediction('02064000', confidence_interval=0.95, start_plot_date='1996-08-31', end_plot_date='1996-09-10')

### Legend

In [15]:
def create_legend(output_dir):
    fig_legend = plt.figure(figsize=(10, 2), dpi=600)
    ax = fig_legend.add_subplot(111)

    color = 'white'
    ax.spines['top'].set_color(color)
    ax.spines['bottom'].set_color(color)
    ax.spines['left'].set_color(color)
    ax.spines['right'].set_color(color)
    ax.tick_params(axis='x', colors=color)
    ax.tick_params(axis='y', colors=color)

    line1, = ax.plot([], [], color='#000000', linewidth=2, label='Observation')
    line2, = ax.plot([], [], color='#E94560', linewidth=2, label='DRUM')
    line3, = ax.plot([], [], color='#0066CC', linewidth=2, label='LSTM-p')
    line4, = ax.plot([], [], color='#16A085', linewidth=2, label='LSTM-d')
    rect1 = plt.Rectangle((0, 0), 1, 1, fc='#E94560', alpha=0.45, label='DRUM 95% central prediction interval ')
    rect2 = plt.Rectangle((0, 0), 1, 1, fc='#0066CC', alpha=0.25, label='LSTM-p 95% central prediction interval')

    legend = ax.legend([line1, line4, line2, line3, rect1, rect2], 
                       ['Observation','LSTM-d', 'DRUM', 'LSTM-p',
                        'DRUM 95% PI', 'LSTM-p 95% PI'], 
                       loc='center', ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.5), labelspacing=1)

    plt.tight_layout(pad=0.1)

    legend_output_file = os.path.join(output_dir, 'Figure_2c_legend.jpg')
    plt.savefig(legend_output_file, dpi=600, bbox_inches='tight', pad_inches=0.05)
    # plt.show()
    plt.close()

create_legend("Figure_2")